# Spatial Clustering Validation: Moran's I

This notebook stages the standalone `SpatialClustering` bundle on a Colab runtime, reuses the `OverallValidation` model-loading code, computes Moran's I for reference and generated BF/BH tiles at epoch 1000, and copies the results back to Google Drive. It is designed for an A100 runtime but also works on smaller GPUs with a lower batch size.

In [ ]:
import subprocess
import sys

# Colab usually has torch/numpy/pandas/matplotlib. Rasterio is the dependency most often missing.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'rasterio'])


0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import tarfile
import time
import zipfile

import torch
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


Mounted at /content/drive
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


## Configure Drive and local staging paths

Before running this notebook, copy both code folders to Drive: `MyDrive/IM3/EvalP1/OverallValidation` and `MyDrive/IM3/EvalP1/SpatialClustering`. The model and archive paths below match the prior Colab validation setup.

In [ ]:
# Drive locations. Edit only if your Drive layout differs.
OVERALL_CODE_DRIVE_DIR = '/content/drive/MyDrive/IM3/EvalP1/OverallValidation'
SPATIAL_CODE_DRIVE_DIR = '/content/drive/MyDrive/IM3/EvalP1/SpatialClustering'
LEGACY_MODEL_DRIVE_ROOT = '/content/drive/MyDrive/IM3/EvalP1/codev3'
MSA_MODEL_DRIVE_ROOT = '/content/drive/MyDrive/IM3/EvalP1/revision_msa_sample'
SAN_DIEGO_TEST_ARCHIVE = '/content/drive/MyDrive/IM3/EvalP1/validation_inputs/Archive_SanDiego_TestNoOverlap.zip'
CONUS_TEST_ARCHIVE = '/content/drive/MyDrive/IM3/EvalP1/validation_inputs/Archive_CONUS_M1_random_stratified_2015_test.zip'
RESULTS_DRIVE_DIR = '/content/drive/MyDrive/IM3/EvalP1/spatial_clustering_results_A100'

# Local Colab staging. These folders are temporary and disappear when the runtime ends.
LOCAL_ROOT = Path('/content/spatial_clustering')
LOCAL_OVERALL_DIR = LOCAL_ROOT / 'OverallValidation'
LOCAL_SPATIAL_DIR = LOCAL_ROOT / 'SpatialClustering'
LOCAL_MODEL_ROOT = LOCAL_ROOT / 'models'
LOCAL_TEST_ROOT = LOCAL_ROOT / 'test_data'

# Runtime settings.
BATCH_SIZE = 128
DEVICE = 'cuda'
RESUME_FULL_RUN = True

ACTIVE_MODEL_FOLDERS = {
    'LALegacy': [
        '1_BF_UNETBaseline',
        '1_BH_UNETBaseline',
        '2A_BF_cGANRandomVecFixed',
        '2A_BH_cGANRandomVecFixed',
        '3_BF_cGANMultiRandomDiversity',
        '3_BH_cGANMultiRandomDiversity',
    ],
    'MSASample': [
        '1_BF_UNETBaseline_MSASample',
        '1_BH_UNETBaseline_MSASample',
        '2A_BF_cGANRandomVecFixed_MSASample',
        '2A_BH_cGANRandomVecFixed_MSASample',
        '3_BF_cGANMultiRandomDiversity_MSASample',
        '3_BH_cGANMultiRandomDiversity_MSASample',
    ],
}


In [ ]:
def resolve_drive_path(raw_path: str | Path) -> Path:
    path = Path(raw_path)
    candidates = [path]
    text = str(path)
    if '/MyDrive/' in text:
        candidates.append(Path(text.replace('/MyDrive/', '/My Drive/')))
    if '/My Drive/' in text:
        candidates.append(Path(text.replace('/My Drive/', '/MyDrive/')))
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not find Drive path. Tried: ' + ', '.join(map(str, candidates)))


def copy_file_if_needed(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        return
    shutil.copy2(src, dst)


def copy_tree_if_missing(src: Path, dst: Path) -> None:
    if dst.exists() and any(dst.iterdir()):
        print(f'Using existing local copy: {dst}')
        return
    print(f'Copying {src} -> {dst}')
    shutil.copytree(src, dst)


def extract_archive(archive_path: Path, extract_dir: Path) -> None:
    extract_dir.mkdir(parents=True, exist_ok=True)
    suffixes = ''.join(archive_path.suffixes).lower()
    print(f'Extracting {archive_path.name} -> {extract_dir}')
    if suffixes.endswith('.zip'):
        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(extract_dir)
    elif suffixes.endswith(('.tar.gz', '.tgz', '.tar')):
        with tarfile.open(archive_path) as tf:
            tf.extractall(extract_dir)
    else:
        raise ValueError(f'Unsupported archive format: {archive_path}')


def find_dataset_root(search_root: Path, required_dirs: list[str]) -> Path:
    candidates = [search_root, *[p for p in search_root.rglob('*') if p.is_dir()]]
    for candidate in candidates:
        if all((candidate / name).is_dir() for name in required_dirs):
            return candidate
    raise FileNotFoundError(f'No dataset root under {search_root} contains {required_dirs}')


def stage_archive_to_local(archive_drive_path: str, local_name: str, required_dirs: list[str]) -> Path:
    drive_archive = resolve_drive_path(archive_drive_path)
    local_archive = LOCAL_ROOT / 'archives' / drive_archive.name
    copy_file_if_needed(drive_archive, local_archive)
    extract_dir = LOCAL_TEST_ROOT / local_name
    if not extract_dir.exists() or not any(extract_dir.iterdir()):
        extract_archive(local_archive, extract_dir)
    root = find_dataset_root(extract_dir, required_dirs)
    print(f'{local_name} dataset root: {root}')
    return root


def count_csv_rows(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open() as handle:
        return max(sum(1 for _ in handle) - 1, 0)


In [ ]:
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

# Copy source files for both bundles. OverallValidation is imported for model/data loading only.
overall_drive_dir = resolve_drive_path(OVERALL_CODE_DRIVE_DIR)
spatial_drive_dir = resolve_drive_path(SPATIAL_CODE_DRIVE_DIR)
LOCAL_OVERALL_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_SPATIAL_DIR.mkdir(parents=True, exist_ok=True)
for name in ['run_overall_validation.py', 'validation_core.py', 'validation_config.json', 'environment.yml']:
    copy_file_if_needed(overall_drive_dir / name, LOCAL_OVERALL_DIR / name)
for name in ['run_spatial_clustering.py', 'spatial_clustering_core.py', 'spatial_clustering_config.json', 'README.md']:
    copy_file_if_needed(spatial_drive_dir / name, LOCAL_SPATIAL_DIR / name)

# Copy only the active model folders needed at epoch 1000.
legacy_drive_root = resolve_drive_path(LEGACY_MODEL_DRIVE_ROOT)
msa_drive_root = resolve_drive_path(MSA_MODEL_DRIVE_ROOT)
local_legacy_root = LOCAL_MODEL_ROOT / 'LALegacyModels'
local_msa_root = LOCAL_MODEL_ROOT / 'MSASampleModels'
for folder in ACTIVE_MODEL_FOLDERS['LALegacy']:
    copy_tree_if_missing(legacy_drive_root / folder, local_legacy_root / folder)
for folder in ACTIVE_MODEL_FOLDERS['MSASample']:
    copy_tree_if_missing(msa_drive_root / folder, local_msa_root / folder)

# Extract held-out test data archives to local disk and infer their actual roots.
san_diego_root = stage_archive_to_local(
    SAN_DIEGO_TEST_ARCHIVE,
    'SanDiegoTestNoOverlap',
    ['central', 'Frac', 'Target'],
)
conus_root = stage_archive_to_local(
    CONUS_TEST_ARCHIVE,
    'CONUSStratifiedTest',
    ['central', 'BFrac', 'BHeight'],
)


Copying /content/drive/MyDrive/IM3/EvalP1/codev3/1_BF_UNETBaseline -> /content/spatial_clustering/models/LALegacyModels/1_BF_UNETBaseline
Copying /content/drive/MyDrive/IM3/EvalP1/codev3/1_BH_UNETBaseline -> /content/spatial_clustering/models/LALegacyModels/1_BH_UNETBaseline
Copying /content/drive/MyDrive/IM3/EvalP1/codev3/2A_BF_cGANRandomVecFixed -> /content/spatial_clustering/models/LALegacyModels/2A_BF_cGANRandomVecFixed
Copying /content/drive/MyDrive/IM3/EvalP1/codev3/2A_BH_cGANRandomVecFixed -> /content/spatial_clustering/models/LALegacyModels/2A_BH_cGANRandomVecFixed
Copying /content/drive/MyDrive/IM3/EvalP1/codev3/3_BF_cGANMultiRandomDiversity -> /content/spatial_clustering/models/LALegacyModels/3_BF_cGANMultiRandomDiversity
Copying /content/drive/MyDrive/IM3/EvalP1/codev3/3_BH_cGANMultiRandomDiversity -> /content/spatial_clustering/models/LALegacyModels/3_BH_cGANMultiRandomDiversity
Copying /content/drive/MyDrive/IM3/EvalP1/revision_msa_sample/1_BF_UNETBaseline_MSASample -> /co

In [ ]:
# Write an OverallValidation config with Colab-local paths. SpatialClustering imports this config.
overall_config = json.loads((LOCAL_OVERALL_DIR / 'validation_config.json').read_text())
overall_config['results_root'] = str(LOCAL_ROOT / 'overall_validation_unused_results')
overall_config['device'] = DEVICE
overall_config['batch_size'] = BATCH_SIZE
overall_config['cache_mode'] = 'lazy'
overall_config['model_roots'] = {
    'LALegacy': str(local_legacy_root),
    'MSASample': str(local_msa_root),
}
overall_config['dataset_specs']['SanDiegoTestNoOverlap']['root'] = str(san_diego_root)
overall_config['dataset_specs']['SanDiegoTestNoOverlap']['manifest'] = str(san_diego_root / 'dataset_manifest.json')
overall_config['dataset_specs']['CONUSStratifiedTest']['root'] = str(conus_root)
overall_config['dataset_specs']['CONUSStratifiedTest']['manifest'] = str(conus_root / 'dataset_manifest.json')
overall_config_path = LOCAL_OVERALL_DIR / 'validation_config_colab.json'
overall_config_path.write_text(json.dumps(overall_config, indent=2) + '\n')

# Write the SpatialClustering config with Colab-local paths.
spatial_config = json.loads((LOCAL_SPATIAL_DIR / 'spatial_clustering_config.json').read_text())
spatial_config['overall_validation_dir'] = str(LOCAL_OVERALL_DIR)
spatial_config['overall_validation_config'] = str(overall_config_path)
spatial_config['results_root'] = str(LOCAL_SPATIAL_DIR / 'results')
spatial_config['device'] = DEVICE
spatial_config['batch_size'] = BATCH_SIZE
spatial_config['cache_mode'] = 'lazy'
spatial_config['checkpoint'] = 1000
spatial_config['targets'] = ['BF', 'BH']
spatial_config['bh_modes'] = ['oracle_bf']
spatial_config['mask_mode'] = 'full_tile'
spatial_config_path = LOCAL_SPATIAL_DIR / 'spatial_clustering_config_colab.json'
spatial_config_path.write_text(json.dumps(spatial_config, indent=2) + '\n')

print('Overall config:', overall_config_path)
print('Spatial config:', spatial_config_path)
print(spatial_config_path.read_text())


Overall config: /content/spatial_clustering/OverallValidation/validation_config_colab.json
Spatial config: /content/spatial_clustering/SpatialClustering/spatial_clustering_config_colab.json
{
  "overall_validation_dir": "/content/spatial_clustering/OverallValidation",
  "overall_validation_config": "/content/spatial_clustering/OverallValidation/validation_config_colab.json",
  "results_root": "/content/spatial_clustering/SpatialClustering/results",
  "device": "cuda",
  "batch_size": 128,
  "cache_mode": "lazy",
  "checkpoint": 1000,
  "training_regimes": [
    "LALegacy",
    "MSASample"
  ],
  "families": [
    "1",
    "2A",
    "3"
  ],
  "datasets": [
    "CONUSStratifiedTest",
    "SanDiegoTestNoOverlap"
  ],
  "learning_rates": [
    0.0001,
    0.0002,
    0.0005,
    0.001
  ],
  "targets": [
    "BF",
    "BH"
  ],
  "bh_modes": [
    "oracle_bf"
  ],
  "primary_latent_seed": 17,
  "mask_mode": "full_tile",
  "neighbor_mode": "rook"
}



In [9]:
cmd = [
    sys.executable,
    str(LOCAL_SPATIAL_DIR / 'run_spatial_clustering.py'),
    'all',
    '--config', str(spatial_config_path),
]
if RESUME_FULL_RUN:
    cmd.append('--resume')

def print_progress() -> None:
    results_root = LOCAL_SPATIAL_DIR / 'results'
    tile_path = results_root / 'metrics' / 'tile_morans_i.csv'
    overall_path = results_root / 'metrics' / 'overall_morans_i.csv'
    log_path = results_root / 'logs' / 'completed_rows.csv'
    print(
        time.strftime('%Y-%m-%d %H:%M:%S'),
        'tile rows', count_csv_rows(tile_path),
        'overall rows', count_csv_rows(overall_path),
        'log', f'{count_csv_rows(log_path)}/96',
    )
    if log_path.exists():
        lines = log_path.read_text().splitlines()
        if len(lines) > 1:
            print('last:', lines[-1])

print('Running:', ' '.join(cmd))
start = time.time()
process = subprocess.Popen(cmd, cwd=str(LOCAL_SPATIAL_DIR))
while process.poll() is None:
    print_progress()
    time.sleep(300)
return_code = process.wait()
print_progress()
elapsed_hours = (time.time() - start) / 3600
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, cmd)
print(f'Spatial clustering validation complete in {elapsed_hours:.2f} hours')


Running: /usr/bin/python3 /content/spatial_clustering/SpatialClustering/run_spatial_clustering.py all --config /content/spatial_clustering/SpatialClustering/spatial_clustering_config_colab.json --resume
2026-07-09 22:44:22 tile rows 0 overall rows 0 log 0/96
2026-07-09 22:49:22 tile rows 12000 overall rows 0 log 6/96
last: 2026-07-09T22:48:47.619277+00:00,moran,moran::BH::oracle_bf::LALegacy::1::CONUSStratifiedTest::lr_0.0005::ckpt_1000,LALegacy::1::CONUSStratifiedTest::lr_0.0005::ckpt_1000,completed
2026-07-09 22:54:22 tile rows 28000 overall rows 0 log 14/96
last: 2026-07-09T22:54:17.326766+00:00,moran,moran::BH::oracle_bf::LALegacy::1::SanDiegoTestNoOverlap::lr_0.0005::ckpt_1000,LALegacy::1::SanDiegoTestNoOverlap::lr_0.0005::ckpt_1000,completed
2026-07-09 22:59:22 tile rows 40000 overall rows 0 log 20/96
last: 2026-07-09T22:58:29.707878+00:00,moran,moran::BH::oracle_bf::LALegacy::2A::CONUSStratifiedTest::lr_0.0002::ckpt_1000,LALegacy::2A::CONUSStratifiedTest::lr_0.0002::ckpt_1000,co

In [10]:
results_root = LOCAL_SPATIAL_DIR / 'results'
for path in [
    results_root / 'metrics' / 'tile_morans_i.csv',
    results_root / 'metrics' / 'overall_morans_i.csv',
    results_root / 'tables' / 'table_morans_i_summary_epoch1000.csv',
    results_root / 'figures' / 'morans_i_model_comparison_epoch1000.png',
]:
    print(path, 'exists=', path.exists(), 'rows=', count_csv_rows(path) if path.suffix == '.csv' else '')


/content/spatial_clustering/SpatialClustering/results/metrics/tile_morans_i.csv exists= True rows= 192000
/content/spatial_clustering/SpatialClustering/results/metrics/overall_morans_i.csv exists= True rows= 96
/content/spatial_clustering/SpatialClustering/results/tables/table_morans_i_summary_epoch1000.csv exists= True rows= 96
/content/spatial_clustering/SpatialClustering/results/figures/morans_i_model_comparison_epoch1000.png exists= True rows= 


In [11]:
results_src = LOCAL_SPATIAL_DIR / 'results'
results_dst = Path(RESULTS_DRIVE_DIR)
results_dst.parent.mkdir(parents=True, exist_ok=True)
if not results_src.exists():
    raise FileNotFoundError(f'Missing local results: {results_src}')
if results_dst.exists():
    shutil.rmtree(results_dst)
print(f'Copying results to Drive: {results_dst}')
shutil.copytree(results_src, results_dst)
print('Results copied to Drive.')


Copying results to Drive: /content/drive/MyDrive/IM3/EvalP1/spatial_clustering_results_A100
Results copied to Drive.
